# Vaidya — Phase 2: QLoRA Training (GCP Vertex AI / RunPod)

**Before running:**
- **GCP Vertex AI Workbench:** GPU: H100 80GB or A100 80GB, PyTorch framework
- **RunPod:** Deploy → A100 40GB → RunPod PyTorch template → Connect → JupyterLab

Fill in your AWS credentials in Step 4 before running.

**Expected runtime:**
- H100 80GB: ~35–45 min (full epoch)
- A100 80GB / 40GB: ~1.5–2 hrs (full epoch)
- L4 24GB: ~2.5 hrs (full epoch)

> train.py auto-detects BF16 on all Ampere/Hopper GPUs.  
> Flash Attention 2 enabled automatically if installed (Step 1).

## Step 1 — Install packages

In [ ]:
# Pinned stable stack
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    bitsandbytes==0.45.3 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3

# Flash Attention 2 — optional, requires nvcc; skip if not available
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn not available — using eager attention (still works)"

print('✅ Install done')

## Step 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU found — check Workbench instance GPU config.'

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability()

print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'CC   : {cc[0]}.{cc[1]}')
print(f'BF16 native : {cc[0] >= 8}')   # True on A100/H100/L4
print(f'Flash Attn  : ', end='')
try:
    import flash_attn; print(f'✅ {flash_attn.__version__}')
except ImportError:
    print('not installed (training still works, just slightly slower)')

## Step 3 — Pull training scripts from GitHub

In [ ]:
import os, shutil

# Auto-detect working directory (RunPod uses /workspace, Vertex AI uses /home/jupyter)
if os.path.exists('/workspace'):
    work_dir = '/workspace'
elif os.path.exists('/home/jupyter'):
    work_dir = '/home/jupyter'
else:
    work_dir = os.path.expanduser('~')

os.chdir(work_dir)
print(f'Working dir: {work_dir}')

!git clone --depth 1 https://github.com/Anand09-in/vaidya.git _repo 2>&1 | tail -2

for f in ['train.py', 'qlora_config.py']:
    shutil.copy(f'_repo/layer2_finetune/{f}', f'./{f}')
    print(f'✅  {f}')

!rm -rf _repo

In [ ]:
import importlib.util, sys, os

work_dir = os.getcwd()
for fname, modname in [('train.py', 'train_mod'), ('qlora_config.py', 'qlora_config')]:
    spec = importlib.util.spec_from_file_location(modname, f'{work_dir}/{fname}')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f"{fname}  →  version {getattr(mod, '__version__', 'MISSING')}")

## Step 4 — AWS credentials

Fill in your AWS keys below. These are only set for this session — never saved to disk.

In [ ]:
import os

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

# Quick check — should list vaidya-artifacts
import boto3
s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
print('Buckets:', buckets)
assert 'vaidya-artifacts' in buckets, 'vaidya-artifacts bucket not found — check credentials'

## Step 5 — Pull data from S3

In [ ]:
import boto3, os, pandas as pd

os.makedirs('./data', exist_ok=True)
s3 = boto3.client('s3')

for page in s3.get_paginator('list_objects_v2').paginate(Bucket='vaidya-artifacts', Prefix='data/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.parquet'):
            fname = key.split('/')[-1]
            local = f'./data/{fname}'
            if not os.path.exists(local):
                print(f'Downloading {key}...')
                s3.download_file('vaidya-artifacts', key, local)

for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'./data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 6 — Pre-flight check

In [ ]:
import importlib

required = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'mlflow', 'boto3']
all_ok = True
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        print(f'✅  {pkg} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'❌  {pkg}  — {e}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Re-run Step 1 and restart the kernel.')
else:
    print('\n✅ Environment ready — proceed to training.')

## Step 7 — Run training

Full epoch — no step cap needed on A100/H100.

In [ ]:
# Dry run — check script loads correctly
!python train.py --help

In [ ]:
# ⚡ FULL TRAINING RUN
# H100: ~35-45 min | A100 80GB: ~60-80 min | L4: ~2.5 hrs
# Change --run-name to run2, run3 etc. for re-runs

!python train.py --run-name run1

## Step 8 — Verify checkpoint on S3

In [ ]:
import boto3

s3 = boto3.client('s3')

print('=== Checkpoint files ===')
resp = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='checkpoints/run1/')
for obj in resp.get('Contents', []):
    print(f"  {obj['Key']}  ({obj['Size']/1e6:.1f} MB)")

print()
print('=== MLflow ===')
resp2 = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='mlflow/')
for obj in resp2.get('Contents', []):
    print(f"  {obj['Key']}")

## Step 9 — Quick eval (loss sanity check)

In [ ]:
import json, glob

states = glob.glob('./checkpoints/run1/checkpoint-*/trainer_state.json')
if not states:
    states = glob.glob('./checkpoints/run1/trainer_state.json')

if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    log_history = state.get('log_history', [])
    train_losses = [(e['step'], e['loss'])      for e in log_history if 'loss'      in e]
    eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]
    if train_losses:
        print(f'Train loss — first: {train_losses[0][1]:.4f}  last: {train_losses[-1][1]:.4f}')
    if eval_losses:
        print(f'Eval  loss — first: {eval_losses[0][1]:.4f}  last: {eval_losses[-1][1]:.4f}')
else:
    print('No trainer_state.json found — check checkpoints/ directory.')

## Gate check — before moving to Phase 3

- ✅ Training loss decreased (not diverged or NaN)
- ✅ Eval loss decreased and didn't spike at the end
- ✅ Checkpoint at `s3://vaidya-artifacts/checkpoints/run1/`
- ✅ MLflow db at `s3://vaidya-artifacts/mlflow/mlflow.db`

**Next:** Phase 3 — accuracy eval (base Mistral-7B vs fine-tuned on MedMCQA test set)